# Blackout Freight · First-shift compass

**Synthetic demonstration** · 2,400 simulated solo attempts. Read the [experiment contract](../docs/first-shift-experiment.md) for enrollment, scope and release criteria.

**Question.** Does an off-screen bearing to the next battery or airlock improve first delivery within 90 seconds? A catch in the same window is the guardrail. The crew subgroup is exploratory.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

analysis_dir = Path.cwd() if Path("compass_report.py").exists() else Path("analysis")
sys.path.insert(0, str(analysis_dir.resolve()))
from compass_report import FIXTURE, build_report, read_counts

report = build_report(read_counts(FIXTURE), "synthetic")
assert report["decision"] == "simulation_only"
solo = report["solo"]
pd.DataFrame(solo["by_variant"]).T[["exposed", "pickups", "deliveries", "catches"]]

## Funnel by segment

Rates use eligible attempts as denominators. Leaving early remains part of the denominator. The catch metric can overlap with successful delivery.

In [ ]:
tables = {name: pd.DataFrame(report[name]["by_variant"]).T for name in ("solo", "crew")}
rates = pd.concat(tables, names=["segment", "variant"])
display(rates[["pickup_rate", "delivery_rate", "catch_rate"]].style.format("{:.1%}"))

## Solo effect and uncertainty

Differences are measured in percentage points. Intervals use the Newcombe method. The dotted reference line shows an equal rate between arms.

In [ ]:
metrics = [("First delivery", solo["delivery"]), ("Caught at least once", solo["catch"])]
fig, ax = plt.subplots(figsize=(8.5, 2.8), layout="constrained")
for index, (label, metric) in enumerate(metrics):
    effect = metric["lift_pp"]
    lower, upper = metric["ci95_pp"]
    ax.errorbar(effect, index, xerr=[[effect - lower], [upper - effect]], fmt="o",
                color=("#006f73", "#915300")[index], capsize=4, linewidth=2)
ax.axvline(0, color="#526574", linestyle=":", linewidth=1.2)
ax.set_yticks(range(len(metrics)), [label for label, _ in metrics])
ax.invert_yaxis()
ax.set_xlabel("Compass − control (percentage points)")
ax.grid(axis="x", alpha=0.15)
ax.spines[["top", "right", "left"]].set_visible(False)
plt.show()

summary = {label: [metric["lift_pp"], *metric["ci95_pp"]] for label, metric in metrics}
display(pd.DataFrame(summary, index=["Effect (pp)", "95% lower", "95% upper"]).T.round(2))
print(f"Allocation check p = {solo['srm_p_value']:.3f}")

## Interpretation

The simulated data show how a result would be evaluated against the predeclared +8-point target and +5-point catch guardrail. No external player outcomes are represented. Open production traffic stays descriptive because one person can contribute multiple attempts and co-op players can affect each other.